In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
base_path = "/content/drive/MyDrive/CustomerSupportAI"

In [8]:
import os
print(os.listdir(base_path))

['emotion_model', 'emotion_label_encoder.pt', 'intent_model', 'intent_label_encoder.pt', 'Emotion dataset']


## Load intent model

In [9]:
intent_label_encoder = torch.load(
    f"{base_path}/intent_label_encoder.pt",
    weights_only=False
)

In [10]:
intent_tokenizer = AutoTokenizer.from_pretrained(f"{base_path}/intent_model")
intent_model = AutoModelForSequenceClassification.from_pretrained(f"{base_path}/intent_model").to(device)
intent_model.eval()

intent_label_encoder = torch.load(
    f"{base_path}/intent_label_encoder.pt",
    weights_only=False
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Load emotion model

In [11]:
emotion_label_encoder = torch.load(
    f"{base_path}/emotion_label_encoder.pt",
    weights_only=False
)

In [12]:
emotion_tokenizer = AutoTokenizer.from_pretrained(f"{base_path}/emotion_model")
emotion_model = AutoModelForSequenceClassification.from_pretrained(f"{base_path}/emotion_model").to(device)
emotion_model.eval()

emotion_label_encoder = torch.load(
    f"{base_path}/emotion_label_encoder.pt",
    weights_only=False
)

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

In [13]:
print("Intent labels:", intent_label_encoder.classes_)
print("Emotion labels:", emotion_label_encoder.classes_)

Intent labels: ['appreciation' 'cancel_request' 'complaint' 'feedback' 'inquiry'
 'refund_replace' 'support_request']
Emotion labels: ['angry' 'confusion' 'frustrated' 'happy' 'neutral' 'sad']


## Intent Prediction

In [14]:
def predict_intent(text):

    inputs = intent_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = intent_model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    pred_id = torch.argmax(probs, dim=1).item()

    return intent_label_encoder.inverse_transform([pred_id])[0]

## Emotion Prediction

In [15]:
def predict_emotion(text):

    inputs = emotion_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = emotion_model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    pred_id = torch.argmax(probs, dim=1).item()

    return emotion_label_encoder.inverse_transform([pred_id])[0]

## Unified analyzer

In [16]:
def analyze_message(text):

    intent = predict_intent(text)
    emotion = predict_emotion(text)

    return {
        "intent": intent,
        "emotion": emotion
    }

In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/CustomerSupportAI"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
analyze_message("I am really frustrated. My refund hasn't arrived yet.")

{'intent': 'refund_replace', 'emotion': 'angry'}

In [20]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ================================
# DEVICE
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================================
# LOAD MODELS
# ================================
base_path = "/content/drive/MyDrive/CustomerSupportAI"

# Intent
intent_tokenizer = AutoTokenizer.from_pretrained(f"{base_path}/intent_model")
intent_model = AutoModelForSequenceClassification.from_pretrained(
    f"{base_path}/intent_model"
).to(device)
intent_model.eval()

intent_label_encoder = torch.load(
    f"{base_path}/intent_label_encoder.pt",
    weights_only=False
)

# Emotion
emotion_tokenizer = AutoTokenizer.from_pretrained(f"{base_path}/emotion_model")
emotion_model = AutoModelForSequenceClassification.from_pretrained(
    f"{base_path}/emotion_model"
).to(device)
emotion_model.eval()

emotion_label_encoder = torch.load(
    f"{base_path}/emotion_label_encoder.pt",
    weights_only=False
)

# ================================
# RESPONSE LOGIC
# ================================

emotion_prefix = {
    "angry": "I completely understand your frustration. ",
    "frustrated": "I can see why this would be frustrating. ",
    "sad": "I'm really sorry you're experiencing this. ",
    "confused": "I understand this might be confusing. ",
    "neutral": "",
    "happy": ""
}

intent_base_response = {
    "complaint": "Let me fix this for you right away.",
    "refund_replace": "I'll check your refund status immediately and update you.",
    "support_request": "I'll guide you step by step to resolve this.",
    "inquiry": "Here’s the information you requested.",
    "appreciation": "We truly appreciate your support!",
    "feedback": "Thank you for helping us improve."
}

# ================================
# CORE PREDICTION ENGINE
# ================================

def predict(model, tokenizer, label_encoder, text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    pred_id = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred_id].item()

    label = label_encoder.inverse_transform([pred_id])[0]

    return label, round(confidence, 4)

# ================================
# MAIN SYSTEM
# ================================

def customer_support_ai(text):

    intent, intent_conf = predict(
        intent_model,
        intent_tokenizer,
        intent_label_encoder,
        text
    )

    emotion, emotion_conf = predict(
        emotion_model,
        emotion_tokenizer,
        emotion_label_encoder,
        text
    )

    prefix = emotion_prefix.get(emotion, "")
    base = intent_base_response.get(intent, "Let me assist you with that.")

    response = prefix + base

    return {
        "intent": intent,
        "intent_confidence": intent_conf,
        "emotion": emotion,
        "emotion_confidence": emotion_conf,
        "response": response
    }

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [20]:
customer_support_ai("I am really frustrated. My refund hasn't arrived yet.")

## Demo

In [21]:
print("="*60)
print("🤖 Customer Support AI Demo")
print("="*60)

test_messages = [
    "I am really frustrated. My refund hasn't arrived yet.",
    "Can you tell me the current interest rate?",
    "Thank you so much for your amazing support!",
    "I can't access my account and this is very annoying.",
    "How do I activate my new card?"
]

for msg in test_messages:

    result = customer_support_ai(msg)

    print("\n------------------------------")
    print("Customer Message:")
    print(msg)

    print("\nPredicted Intent:", result["intent"])
    print("Intent Confidence:", result["intent_confidence"])

    print("Predicted Emotion:", result["emotion"])
    print("Emotion Confidence:", result["emotion_confidence"])

    print("\nAI Response:")
    print(result["response"])

🤖 Customer Support AI Demo

------------------------------
Customer Message:
I am really frustrated. My refund hasn't arrived yet.

Predicted Intent: refund_replace
Intent Confidence: 0.9976
Predicted Emotion: angry
Emotion Confidence: 0.545

AI Response:
I completely understand your frustration. I'll check your refund status immediately and update you.

------------------------------
Customer Message:
Can you tell me the current interest rate?

Predicted Intent: inquiry
Intent Confidence: 0.9977
Predicted Emotion: neutral
Emotion Confidence: 0.9507

AI Response:
Here’s the information you requested.

------------------------------
Customer Message:
Thank you so much for your amazing support!

Predicted Intent: appreciation
Intent Confidence: 0.9977
Predicted Emotion: happy
Emotion Confidence: 0.5557

AI Response:
We truly appreciate your support!

------------------------------
Customer Message:
I can't access my account and this is very annoying.

Predicted Intent: complaint
Intent C

In [ ]:
from google.colab import _message
from google.colab import files
import json

# Get notebook JSON
nb = _message.blocking_request('get_ipynb', timeout_sec=5)

# Extract notebook name safely
metadata = nb.get('ipynb', {}).get('metadata', {})
colab_meta = metadata.get('colab', {})

nb_name = colab_meta.get('name', 'downloaded_notebook_unified.ipynb')

print("Notebook name:", nb_name)

# Save notebook to runtime
with open(nb_name, 'w') as f:
    json.dump(nb['ipynb'], f)

# Download to your computer
files.download(nb_name)